In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

_START_TIME = time.time()
BOOTSTRAP_PACKAGES = ["pyyaml==6.0.3", "huggingface_hub==1.13.0"]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *BOOTSTRAP_PACKAGES])

import yaml
from huggingface_hub import HfApi, hf_hub_download


def resolve_hf_token():
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient

        secrets = UserSecretsClient()
        return secrets.get_secret("HF_TOKEN")
    except Exception:
        return None


def require_config(config):
    required_sections = [
        "environment",
        "repos",
        "model",
        "lora",
        "training",
        "collection",
        "evaluation",
        "data_sources",
    ]
    missing = [section for section in required_sections if section not in config]
    if missing:
        raise RuntimeError(f"SFT config missing sections: {missing}")
    packages = config["environment"].get("pip_packages", [])
    if not packages or any("==" not in package for package in packages):
        raise RuntimeError("Kaggle pip packages must be exact pins in sft_05b.yaml.")
    train_cfg = config["training"]
    if train_cfg.get("fp16") is not True or train_cfg.get("bf16") is not False:
        raise RuntimeError("Kaggle P100 config must use fp16=True and bf16=False.")


HF_TOKEN = resolve_hf_token()
if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN as a Kaggle secret before running this notebook.")

api = HfApi(token=HF_TOKEN)
whoami = api.whoami(token=HF_TOKEN)
HF_USER = whoami.get("name")
if not isinstance(HF_USER, str) or not HF_USER:
    raise RuntimeError("Could not resolve Hugging Face username from HF_TOKEN.")
DATASET_REPO = f"{HF_USER}/dataforge-sft-trajectories"
CONFIG_PATH = Path(
    hf_hub_download(
        repo_id=DATASET_REPO, filename="sft_05b.yaml", repo_type="dataset", token=HF_TOKEN
    )
)
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
require_config(config)

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *config["environment"]["pip_packages"]]
)
MODEL_REPO = config["repos"]["model_repo_template"].format(hf_user=HF_USER)
print(f"Resolved HF dataset repo: {DATASET_REPO}")
print(f"Resolved HF model repo: {MODEL_REPO}")

In [ ]:
import json

from datasets import load_dataset

DATA_PATH = Path(
    hf_hub_download(
        repo_id=DATASET_REPO,
        filename=config["repos"]["trajectory_filename"],
        repo_type="dataset",
        token=HF_TOKEN,
    )
)
CARD_TEMPLATE_PATH = Path(
    hf_hub_download(
        repo_id=DATASET_REPO,
        filename=config["repos"]["model_card_template_filename"],
        repo_type="dataset",
        token=HF_TOKEN,
    )
)


def validate_trajectory_dataset(dataset, cfg):
    record_count = len(dataset)
    if record_count < 2:
        raise RuntimeError(
            f"Need at least 2 SFT records for a non-empty train/held-out split; found {record_count}."
        )
    expected_schema = cfg["collection"]["schema_version"]
    expected_teacher = cfg["collection"]["teacher_model"]
    min_episode_f1 = float(cfg["collection"]["min_episode_f1"])
    seen_ids = set()
    seen_keys = set()
    for index, record in enumerate(dataset, start=1):
        if record.get("schema_version") != expected_schema:
            raise RuntimeError(f"Record {index} has wrong schema_version: {record.get('schema_version')!r}")
        trajectory_id = record.get("trajectory_id")
        if trajectory_id in seen_ids:
            raise RuntimeError(f"Duplicate trajectory_id in HF dataset: {trajectory_id}")
        seen_ids.add(trajectory_id)
        key = (record.get("task_id"), int(record.get("seed", -1)), int(record.get("chunk_index", -1)))
        if key in seen_keys:
            raise RuntimeError(f"Duplicate SFT chunk key in HF dataset: {key}")
        seen_keys.add(key)
        metrics = record.get("metrics") or {}
        if float(metrics.get("episode_f1", -1.0)) < min_episode_f1:
            raise RuntimeError(f"Record {index} is below the configured episode F1 floor.")
        teacher = record.get("teacher") or {}
        if teacher.get("model") != expected_teacher:
            raise RuntimeError(f"Record {index} teacher metadata does not match sft_05b.yaml.")
        if not record.get("messages"):
            raise RuntimeError(f"Record {index} has no chat messages.")
    test_size = min(100, max(1, record_count // 10))
    if record_count - test_size < 1:
        raise RuntimeError("SFT split would leave no training records.")
    return test_size


raw_dataset = load_dataset("json", data_files=str(DATA_PATH), split="train")
test_size = validate_trajectory_dataset(raw_dataset, config)
split = raw_dataset.train_test_split(test_size=test_size, seed=42)
train_records = split["train"]
heldout_records = split["test"]
print(
    f"Loaded {len(train_records)} training records and {len(heldout_records)} local holdout records from expert_v1.jsonl"
)

In [ ]:
import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU runtime is required; enable a P100 or compatible CUDA GPU.")
gpu_name = torch.cuda.get_device_name(0)
print(f"CUDA GPU: {gpu_name}")

quant_cfg = config["model"]["quantization"]
bnb_config = BitsAndBytesConfig(
    load_in_4bit=quant_cfg["load_in_4bit"],
    bnb_4bit_quant_type=quant_cfg["bnb_4bit_quant_type"],
    bnb_4bit_use_double_quant=quant_cfg["bnb_4bit_use_double_quant"],
    bnb_4bit_compute_dtype=torch.float16,
)

base_model_id = config["model"]["base_model"]
tokenizer = AutoTokenizer.from_pretrained(
    base_model_id, token=HF_TOKEN, trust_remote_code=config["model"]["trust_remote_code"]
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=config["model"]["trust_remote_code"],
)
model.config.use_cache = False

lora = config["lora"]
peft_config = LoraConfig(
    r=lora["r"],
    lora_alpha=lora["alpha"],
    lora_dropout=lora["dropout"],
    bias=lora["bias"],
    task_type=lora["task_type"],
    target_modules=lora["target_modules"],
)

In [ ]:
import gc

from trl import SFTConfig, SFTTrainer


def render_chat_text(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"], tokenize=False, add_generation_prompt=False
        )
    }


train_dataset = train_records.map(render_chat_text, remove_columns=train_records.column_names)


def latest_checkpoint_path(output_dir: Path):
    checkpoints = sorted(
        output_dir.glob("checkpoint-*"), key=lambda path: int(path.name.split("-")[-1])
    )
    return str(checkpoints[-1]) if checkpoints else None


train_cfg = config["training"]
output_dir = Path(train_cfg["output_dir"])
training_args = SFTConfig(
    output_dir=str(output_dir),
    num_train_epochs=train_cfg["num_train_epochs"],
    per_device_train_batch_size=train_cfg["per_device_train_batch_size"],
    gradient_accumulation_steps=train_cfg["gradient_accumulation_steps"],
    learning_rate=train_cfg["learning_rate"],
    lr_scheduler_type=train_cfg["lr_scheduler_type"],
    warmup_ratio=train_cfg["warmup_ratio"],
    weight_decay=train_cfg["weight_decay"],
    max_length=train_cfg["max_seq_length"],
    packing=train_cfg["packing"],
    logging_steps=train_cfg["logging_steps"],
    save_steps=train_cfg["save_steps"],
    save_total_limit=train_cfg["save_total_limit"],
    fp16=train_cfg["fp16"],
    bf16=train_cfg["bf16"],
    gradient_checkpointing=train_cfg["gradient_checkpointing"],
    report_to=train_cfg["report_to"],
    dataset_text_field="text",
)
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)
latest_checkpoint = latest_checkpoint_path(output_dir)
trainer.train(resume_from_checkpoint=latest_checkpoint)
trainer.save_model(train_cfg["adapter_dir"])
tokenizer.save_pretrained(train_cfg["adapter_dir"])
del trainer, model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from peft import PeftModel

adapter_dir = Path(config["training"]["adapter_dir"])
merged_dir = Path(config["training"]["merged_dir"])
if not adapter_dir.exists():
    raise RuntimeError(f"Missing trained adapter directory: {adapter_dir}")
base_for_merge = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=config["model"]["trust_remote_code"],
)
merged = PeftModel.from_pretrained(base_for_merge, adapter_dir)
merged = merged.merge_and_unload()
merged.save_pretrained(merged_dir, safe_serialization=True)
tokenizer.save_pretrained(merged_dir)
del merged, base_for_merge
gc.collect()
torch.cuda.empty_cache()
print(f"Merged LoRA adapter into {merged_dir}")

In [ ]:
import hashlib
import time

import pandas as pd


def load_bench_dataset(name):
    source = config["data_sources"][name]
    dirty = pd.read_csv(source["dirty_url"], dtype=str, keep_default_na=False, na_filter=False)
    clean = pd.read_csv(source["clean_url"], dtype=str, keep_default_na=False, na_filter=False)
    if len(dirty.index) != len(clean.index) or len(dirty.columns) != len(clean.columns):
        raise RuntimeError(f"Dirty/clean shape mismatch for held-out dataset {name}.")
    dirty.columns = [str(column) for column in clean.columns]
    ground_truth = []
    for row_idx, (dirty_row, clean_row) in enumerate(
        zip(dirty.itertuples(index=False, name=None), clean.itertuples(index=False, name=None), strict=True)
    ):
        for column, dirty_value, clean_value in zip(clean.columns, dirty_row, clean_row, strict=True):
            if str(dirty_value) != str(clean_value):
                ground_truth.append({"row": row_idx, "column": str(column), "clean_value": str(clean_value)})
    return dirty, tuple(str(column) for column in clean.columns), ground_truth


def chunk_rows(df, columns, truth, seed, width=8):
    width = min(width, len(df.index))
    if width <= 0:
        return []
    digest = hashlib.sha256(str(seed).encode("utf-8")).hexdigest()
    if truth:
        anchor = int(truth[int(digest[:8], 16) % len(truth)]["row"])
        start = min(max(anchor - width // 2, 0), len(df.index) - width)
    else:
        start = int(digest[:8], 16) % max(1, len(df.index) - width + 1)
    rows = []
    for row_idx in range(start, start + width):
        row = {"_row": row_idx}
        for column in columns:
            row[column] = str(df.iloc[row_idx][column])
        rows.append(row)
    return rows


def parse_repairs(text):
    decoder = json.JSONDecoder()
    for offset, char in enumerate(text):
        if char != "{":
            continue
        try:
            payload, _ = decoder.raw_decode(text[offset:])
        except json.JSONDecodeError:
            continue
        repairs = payload.get("repairs", []) if isinstance(payload, dict) else []
        return [repair for repair in repairs if isinstance(repair, dict)]
    return []


def f1_score(truth, repairs):
    truth_map = {(cell["row"], cell["column"]): cell["clean_value"] for cell in truth}
    predictions = {}
    for repair in repairs:
        if {"row", "column", "new_value"}.issubset(repair):
            try:
                key = (int(repair["row"]), str(repair["column"]))
            except (TypeError, ValueError):
                continue
            predictions[key] = str(repair["new_value"])
    tp = sum(1 for key, value in predictions.items() if truth_map.get(key) == value)
    fp = sum(1 for key in predictions if key not in truth_map or truth_map[key] != predictions[key])
    fn = len(set(truth_map) - {key for key, value in predictions.items() if truth_map.get(key) == value})
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return 2 * precision * recall / (precision + recall) if precision + recall else 0.0


def build_eval_tasks():
    target = int(config["evaluation"]["heldout_tasks"])
    datasets = {name: load_bench_dataset(name) for name in config["evaluation"]["datasets"]}
    tasks = []
    seed = int(config["evaluation"]["seeds_start"])
    attempts = 0
    max_attempts = max(target * 20, 20)
    while len(tasks) < target and attempts < max_attempts:
        for dataset_name, (dirty, columns, truth) in datasets.items():
            rows = chunk_rows(dirty, columns, truth, seed + attempts)
            attempts += 1
            row_ids = {row["_row"] for row in rows}
            task_truth = [cell for cell in truth if cell["row"] in row_ids]
            if not task_truth:
                continue
            tasks.append(
                {
                    "schema_summary": {"dataset": dataset_name, "columns": list(columns)},
                    "rows": rows,
                    "ground_truth": task_truth,
                }
            )
            if len(tasks) >= target or attempts >= max_attempts:
                break
    if len(tasks) < target:
        raise RuntimeError(f"Could only build {len(tasks)} held-out tasks; requested {target}.")
    return tasks


def evaluate_model(eval_model, eval_tokenizer, tasks):
    eval_model.eval()
    device = next(eval_model.parameters()).device
    scores = []
    for task in tasks:
        prompt_messages = [
            {"role": "system", "content": "Return strict JSON with a repairs list for dirty tabular rows."},
            {
                "role": "user",
                "content": json.dumps({"schema_summary": task["schema_summary"], "rows": task["rows"]}, sort_keys=True),
            },
        ]
        prompt = eval_tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
        inputs = eval_tokenizer(prompt, return_tensors="pt").to(device)
        with torch.inference_mode():
            output = eval_model.generate(
                **inputs,
                max_new_tokens=config["evaluation"]["max_new_tokens"],
                do_sample=False,
                pad_token_id=eval_tokenizer.eos_token_id,
            )
        decoded = eval_tokenizer.decode(output[0][inputs["input_ids"].shape[-1] :], skip_special_tokens=True)
        scores.append(f1_score(task["ground_truth"], parse_repairs(decoded)))
    return round(sum(scores) / len(scores), 4) if scores else 0.0


def evaluate_checkpoint(model_ref, *, token=None):
    eval_model = AutoModelForCausalLM.from_pretrained(
        model_ref,
        torch_dtype=torch.float16,
        device_map="auto",
        token=token,
        trust_remote_code=config["model"]["trust_remote_code"],
    )
    score = evaluate_model(eval_model, tokenizer, tasks)
    del eval_model
    gc.collect()
    torch.cuda.empty_cache()
    return score


tasks = build_eval_tasks()
base_f1 = evaluate_checkpoint(base_model_id, token=HF_TOKEN)
sft_f1 = evaluate_checkpoint(merged_dir)
training_metrics = {
    "model_name": "DataForge-0.5B-SFT",
    "model_license": config["model"]["model_license"],
    "base_model": base_model_id,
    "teacher_model": config["collection"]["teacher_model"],
    "dataset_repo": DATASET_REPO,
    "training_examples": len(train_records),
    "kaggle_hours": round((time.time() - globals().get("_START_TIME", time.time())) / 3600, 3),
    "base_f1": base_f1,
    "sft_f1": sft_f1,
    "repo_id": MODEL_REPO,
}
(merged_dir / "training_metrics.json").write_text(json.dumps(training_metrics, indent=2), encoding="utf-8")
(merged_dir / "README.md").write_text(
    CARD_TEMPLATE_PATH.read_text(encoding="utf-8").format(**training_metrics), encoding="utf-8"
)
api.create_repo(repo_id=MODEL_REPO, repo_type="model", exist_ok=True, token=HF_TOKEN)
api.upload_folder(
    folder_path=str(merged_dir),
    repo_id=MODEL_REPO,
    repo_type="model",
    token=HF_TOKEN,
    commit_message="Publish Week 9 DataForge 0.5B SFT checkpoint with evaluation metrics",
)
print(f"Base F1: {base_f1}")
print(f"SFT F1: {sft_f1}")
print(f"Pushed merged checkpoint to {MODEL_REPO}")